# Imports

In [0]:
from pyspark.sql.functions import trim, when, length, lit, col, row_number, lower as lower_spark, concat_ws, coalesce, current_timestamp, sha2, sum as sum_spark, lower as lower_spark, upper as upper_spark, countDistinct, first, dense_rank
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
CATALOG = "workspace"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

BRONZE_QUALIFYING_RESULTS_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.qualifying_results"
)

SILVER_QUALIFYING_RESULTS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.qualifying_results"
)

SILVER_RACES_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.races"
)

SILVER_DRIVERS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.drivers"
)

SILVER_CONSTRUCTORS_TABLE = (
    f"{CATALOG}.{SILVER_SCHEMA}.constructors"
)

# Metodos

In [0]:
def clean_string(column_name: str):
    value = trim(col(column_name))

    return (
        when(length(value) == 0, lit(None).cast("string"))
         .otherwise(value)
    )

In [0]:
def ns_to_ms(column_name: str):
    return (
        when(
            col(column_name).isNotNull(),
            (col(column_name) / 1_000_000).cast("long")
        )
    )

In [0]:
def get_latest_qualifying_results_snapshot():
    bronze_df = spark.table(
        BRONZE_QUALIFYING_RESULTS_TABLE
    )

    snapshot_window = (
        Window
        .partitionBy(
            "season",
            "round"
        )
        .orderBy(
            col("_source_file_modification_time")
                .desc_nulls_last(),

            col("_ingested_at")
                .desc_nulls_last()
        )
    )

    return (
        bronze_df
        .withColumn(
            "_snapshot_rank",
            dense_rank().over(snapshot_window)
        )
        .filter(
            col("_snapshot_rank") == 1
        )
        .drop("_snapshot_rank")
    )

In [0]:
def transform_qualifying_results(df):
    position_window = (
        Window
        .partitionBy("season", "round")
        .orderBy(
            when(col("Q3").isNotNull(), lit(3))
            .when(col("Q2").isNotNull(), lit(2))
            .when(col("Q1").isNotNull(), lit(1))
            .otherwise(lit(0))
            .desc(),
            coalesce(col("Q3"), col("Q2"), col("Q1"))
            .asc_nulls_last()
        )
    )

    return (
        df
        .withColumn(
            "_qualifying_position",
            row_number().over(position_window)
        )
        .select(
            # ----------------------------------------------
            # Race
            # ----------------------------------------------

            col("season")
                .cast("int")
                .alias("season"),

            col("round")
                .cast("int")
                .alias("round"),

            # ----------------------------------------------
            # Driver / constructor
            # ----------------------------------------------

            lower_spark(
                clean_string("DriverId")
            ).alias("driver_id"),

            clean_string("DriverNumber")
                .alias("driver_number"),

            lower_spark(
                clean_string("TeamId")
            ).alias("constructor_id"),

            # ----------------------------------------------
            # Qualifying result
            # ----------------------------------------------

            col("_qualifying_position")
                .cast("int")
                .alias("qualifying_position"),

            ns_to_ms("Q1")
                .alias("q1_time_ms"),

            ns_to_ms("Q2")
                .alias("q2_time_ms"),

            ns_to_ms("Q3")
                .alias("q3_time_ms"),

            # ----------------------------------------------
            # Lineage
            # ----------------------------------------------

            col("_source_file")
                .alias("source_file"),

            col("_source_file_modification_time")
                .alias("source_modified_at"),

            col("_ingested_at")
                .alias("bronze_ingested_at")
        )
    )

In [0]:
def validate_qualifying_results_references(df):
    errors = {}

    # ------------------------------------------------------
    # Race
    # ------------------------------------------------------

    valid_races = (
        spark.table(SILVER_RACES_TABLE)
        .select(
            "season",
            "round"
        )
        .distinct()
    )

    missing_races = (
        df
        .select(
            "season",
            "round"
        )
        .distinct()
        .join(
            valid_races,
            ["season", "round"],
            "left_anti"
        )
        .count()
    )

    if missing_races > 0:
        errors["missing_race"] = missing_races

    # ------------------------------------------------------
    # Driver
    # ------------------------------------------------------

    valid_drivers = (
        spark.table(SILVER_DRIVERS_TABLE)
        .select("driver_id")
        .distinct()
    )

    missing_drivers = (
        df
        .select("driver_id")
        .distinct()
        .join(
            valid_drivers,
            ["driver_id"],
            "left_anti"
        )
        .count()
    )

    if missing_drivers > 0:
        errors["missing_driver"] = missing_drivers

    # ------------------------------------------------------
    # Constructor
    # ------------------------------------------------------

    valid_constructors = (
        spark.table(SILVER_CONSTRUCTORS_TABLE)
        .select("constructor_id")
        .distinct()
    )

    missing_constructors = (
        df
        .select("constructor_id")
        .distinct()
        .join(
            valid_constructors,
            ["constructor_id"],
            "left_anti"
        )
        .count()
    )

    if missing_constructors > 0:
        errors["missing_constructor"] = (
            missing_constructors
        )

    if errors:
        raise ValueError(
            f"Qualifying referential validation failed: {errors}"
        )

    print(
        "Qualifying referential validation OK."
    )

In [0]:
def validate_qualifying_results(df):
    validation = (
        df
        .agg(
            sum_spark(
                when(
                    col("season").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_season"),

            sum_spark(
                when(
                    col("round").isNull()
                    |
                    (col("round") <= 0),
                    1
                ).otherwise(0)
            ).alias("invalid_round"),

            sum_spark(
                when(
                    col("driver_id").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_driver_id"),

            sum_spark(
                when(
                    col("constructor_id").isNull(),
                    1
                ).otherwise(0)
            ).alias("null_constructor_id"),

            sum_spark(
                when(
                    col("qualifying_position").isNull()
                    |
                    (col("qualifying_position") <= 0),
                    1
                ).otherwise(0)
            ).alias("invalid_qualifying_position"),

            sum_spark(
                when(
                    col("q1_time_ms") < 0,
                    1
                ).otherwise(0)
            ).alias("negative_q1"),

            sum_spark(
                when(
                    col("q2_time_ms") < 0,
                    1
                ).otherwise(0)
            ).alias("negative_q2"),

            sum_spark(
                when(
                    col("q3_time_ms") < 0,
                    1
                ).otherwise(0)
            ).alias("negative_q3")
        )
        .first()
        .asDict()
    )

    errors = {
        rule: value or 0
        for rule, value in validation.items()
        if (value or 0) > 0
    }

    # Natural key
    duplicate_keys = (
        df
        .groupBy(
            "season",
            "round",
            "driver_id"
        )
        .count()
        .filter(
            col("count") > 1
        )
        .count()
    )

    if duplicate_keys > 0:
        errors["duplicate_race_driver"] = duplicate_keys

    # Qualifying position should also be unique
    duplicate_positions = (
        df
        .filter(
            col("qualifying_position").isNotNull()
        )
        .groupBy(
            "season",
            "round",
            "qualifying_position"
        )
        .count()
        .filter(
            col("count") > 1
        )
        .count()
    )

    if duplicate_positions > 0:
        errors["duplicate_qualifying_position"] = (
            duplicate_positions
        )

    if errors:
        raise ValueError(
            f"Silver qualifying validation failed: {errors}"
        )

    print(
        f"Validation OK: {df.count()} "
        "qualifying results ready for Silver."
    )

In [0]:
def show_qualifying_time_anomalies(df):
    anomalies = (
        df
        .filter(
            (
                col("q2_time_ms").isNotNull()
                &
                col("q1_time_ms").isNull()
            )
            |
            (
                col("q3_time_ms").isNotNull()
                &
                col("q2_time_ms").isNull()
            )
        )
    )

    anomaly_count = anomalies.count()

    if anomaly_count > 0:
        print(
            f"WARNING: {anomaly_count} qualifying rows "
            "have unexpected Q1/Q2/Q3 progression."
        )

        display(anomalies)

In [0]:
def add_qualifying_results_hash(df):
    business_columns = [
        "season",
        "round",
        "driver_id",
        "driver_number",
        "constructor_id",
        "qualifying_position",
        "q1_time_ms",
        "q2_time_ms",
        "q3_time_ms"
    ]

    hash_expression = concat_ws(
        "||",
        *[
            coalesce(
                col(column).cast("string"),
                lit("<NULL>")
            )
            for column in business_columns
        ]
    )

    return (
        df
        .withColumn(
            "record_hash",
            sha2(
                hash_expression,
                256
            )
        )
        .withColumn(
            "silver_updated_at",
            current_timestamp()
        )
    )

In [0]:
def merge_qualifying_results(df):
    if not spark.catalog.tableExists(
        SILVER_QUALIFYING_RESULTS_TABLE
    ):
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(
                SILVER_QUALIFYING_RESULTS_TABLE
            )
        )

        print(
            f"Created {SILVER_QUALIFYING_RESULTS_TABLE}"
        )
        return

    target = DeltaTable.forName(
        spark,
        SILVER_QUALIFYING_RESULTS_TABLE
    )

    (
        target.alias("target")
        .merge(
            df.alias("source"),
            """
            target.season = source.season
            AND target.round = source.round
            AND target.driver_id = source.driver_id
            """
        )
        .whenMatchedUpdateAll(
            condition="""
                target.record_hash <> source.record_hash
            """
        )
        .whenNotMatchedInsertAll()

        # Safe here because df represents the complete
        # latest state contained in Bronze.
        .whenNotMatchedBySourceDelete()

        .execute()
    )

    print(
        f"Merged data into {SILVER_QUALIFYING_RESULTS_TABLE}"
    )

In [0]:
qualifying_source_df = (
    get_latest_qualifying_results_snapshot()
)

qualifying_results_df = (
    transform_qualifying_results(
        qualifying_source_df
    )
)

validate_qualifying_results_references(
    qualifying_results_df
)

validate_qualifying_results(
    qualifying_results_df
)

show_qualifying_time_anomalies(
    qualifying_results_df
)

qualifying_results_df = (
    add_qualifying_results_hash(
        qualifying_results_df
    )
)

merge_qualifying_results(
    qualifying_results_df
)